In [ ]:
pip install python-dotenv

In [45]:
import pandas as pd
import json
import requests
import os # no hay que instalarlo
import time
from dotenv import load_dotenv # esta es una librería para poder encriptar nuestra api_key

In [ ]:
# leer el json
df = pd.read_json('definitivo.json')
df

,id,artist_name,track_name,genre,year
0,7AlYBA5M9FVXDqN31cbExE,Remedios Amaya,El zarandeo,flamenco,2000
1,4uqcES2h19TKAXQlciycrO,Nobuo Uematsu,Vamo'alla flamenco,flamenco,2000
2,0z6xGK2u3nSsbk9XeKYwZQ,Maurice Ravel,Flamenco Bolero,flamenco,2000
3,6DjfWx0fXi0uQpedpvaCD9,Mark Towns,After the Rain,flamenco,2000
4,1t91qQaPbOoo4pumaLih4W,Franz Schubert,Tango Serenata (Ständchen),flamenco,2000
...,...,...,...,...,...
61612,6Lv0uLFhPt6e4qcmoeltAq,Jade Macrae,Ain’t Nobody Home,rock,2026
61613,6Lv0uLFhPt6e4qcmoeltAq,Robben Ford,Ain’t Nobody Home,rock,2026
61614,4PtUbfBR2U2BC0uUrT9hnM,No Te Va Gustar,Todo Mal,rock,2026
61615,4PtUbfBR2U2BC0uUrT9hnM,Ciro y los Persas,Todo Mal,rock,2026


In [23]:
df_filtrado = df[
    (df['genre'] == 'rock') &
    (df['year'].between(2013, 2015))
]

df_filtrado


,id,artist_name,track_name,genre,year
28406,1hvf4GP5GbVQ8aPbVXIsEW,McFly,McFly The Musical - Live At The Royal Albert Hall,rock,2013
28407,4ApSXhiRpboS6M8e0kg9Jq,Tess,The Second That You Sleep - Interphace Remix,rock,2013
28408,36rH4mPe87uobaXzb5uPNG,Midiland,Renegade Dancers - Version Dance,rock,2013
28409,1KqCw7AwVIgRJAIBgH1poL,Los Flechazos,Suzette,rock,2013
28410,318HbfUmnbbZ9C0wmF4Hf0,Kamembert,Sha la la,rock,2013
...,...,...,...,...,...
33602,4jK6bBO2JbkvvgT7FPj8c5,Kurt Cobain,Poison's Gone,rock,2015
33603,156OOIRuzypvxOGEYbo4gO,Hombres G,El ataque de las chicas cocodrilo - Las Ventas...,rock,2015
33604,6x6HKeycSp85NcIaTRkoi6,Don Osvaldo,Mis Latidos,rock,2015
33605,0DxG8MVe9qRQG1gc7LIObk,The Strypes,Eighty-Four,rock,2015


In [ ]:
# Guardamos los datos filtrados en un JSON 

df_filtrado.to_json(
    'rock_2013_2015.json',
    orient='records',       # lo guarda como lista de diccionarios
    indent=4,               # añade sangría (espacios) para que sea fácil de leer.
    force_ascii=False       # mantiene tildes y caracteres especiales correctamente.
)



LAST_FM

In [40]:
import pandas as pd

df = pd.read_json('rock_2013_2015.json')

df.head()


,id,artist_name,track_name,genre,year
0,1hvf4GP5GbVQ8aPbVXIsEW,McFly,McFly The Musical - Live At The Royal Albert Hall,rock,2013
1,4ApSXhiRpboS6M8e0kg9Jq,Tess,The Second That You Sleep - Interphace Remix,rock,2013
2,36rH4mPe87uobaXzb5uPNG,Midiland,Renegade Dancers - Version Dance,rock,2013
3,1KqCw7AwVIgRJAIBgH1poL,Los Flechazos,Suzette,rock,2013
4,318HbfUmnbbZ9C0wmF4Hf0,Kamembert,Sha la la,rock,2013


In [46]:
load_dotenv()   # Carga el archivo .env

True

In [47]:
api_key = os.getenv("API_KEY") # accedemos a la variable api_key que está en el archivo .env

print(api_key)

7d042a85593c44c0449486979b28b055


In [48]:
detalles_artistas=[]

In [49]:
df['artist_name'] = df['artist_name'].str.strip()

In [50]:
#  Recorrer artistas y consultar API
for artist in df['artist_name'].unique():
    params = {
        'method': 'artist.getInfo',
        'artist': artist,
        'api_key': api_key,
        'format': 'json'
    }
    
    try:
        response = requests.get('http://ws.audioscrobbler.com/2.0/', params=params)
        data = response.json()
        
        if 'artist' in data:
            info = {
                'artist_name': artist,
                'listeners': data['artist']['stats']['listeners'],
                'playcount': data['artist']['stats']['playcount'],
                'bio': data['artist']['bio']['summary'],
                'url': data['artist']['url'],
                'tags': [tag['name'] for tag in data['artist']['tags']['tag']]
            }
            
            detalles_artistas.append(info)
        
        time.sleep(0.2)  # evitar límite de peticiones

    except Exception as e:
        print(f"Error con {artist}: {e}")


In [51]:
# Convertir a DataFrame
df_detalles = pd.DataFrame(detalles_artistas)

In [52]:
df_detalles

,artist_name,listeners,playcount,bio,url,tags
0,McFly,798487,47104987,McFly are a British pop rock band who first fo...,https://www.last.fm/music/McFly,"[pop rock, british, rock, pop, britpop]"
1,Tess,68864,474246,There are currently 5 acts listed under the so...,https://www.last.fm/music/Tess,"[post-hardcore, screamo, french, hardcore, eur..."
2,Midiland,61,225,"<a href=""https://www.last.fm/music/Midiland"">...",https://www.last.fm/music/Midiland,[]
3,Los Flechazos,16775,191525,"<a href=""https://www.last.fm/music/Los+Flecha...",https://www.last.fm/music/Los+Flechazos,"[mod, mod revival, spanish, spanish indie pop,..."
4,Kamembert,373,1337,"<a href=""https://www.last.fm/music/Kamembert""...",https://www.last.fm/music/Kamembert,[]
...,...,...,...,...,...,...
391,Jacco Gardner,150710,2061150,"Jacco Gardner [born 9 April 1988 in Hoorn, The...",https://www.last.fm/music/Jacco+Gardner,"[psychedelic, psychedelic pop, baroque pop, po..."
392,Oh Hiroshima,97467,1732173,Founded over 15 years ago as a DIY post-rock r...,https://www.last.fm/music/Oh+Hiroshima,"[post-rock, instrumental, experimental, Sludge..."
393,Little May,228841,1389277,Little May is a pop/folk/indie band formed in ...,https://www.last.fm/music/Little+May,"[indie pop, folk, australian, indie, pop]"
394,The Strypes,141150,1702323,The Strypes were a 4-piece rock band from Cava...,https://www.last.fm/music/The+Strypes,"[indie rock, irish, blues, rock, blues rock]"


In [53]:
# Guardar en JSON 
df_detalles.to_json(
    'detalles_artistas_rock.json',
    orient='records',
    indent=4,
    force_ascii=False

)

print("Datos de artistas guardados en 'detalles_artistas_rock.json'")

Datos de artistas guardados en 'detalles_artistas_rock.json'


In [54]:

# Leer el JSON
df = pd.read_json('detalles_artistas_rock.json')
df = pd.read_json('rock_2013_2015.json')

# Guardarlo como CSV
df.to_csv('detalles_artistas_rock.csv', index=False)
df.to_csv('rock_2013_2015.csv', index=False)

print("Archivo convertido a CSV correctamente")


Archivo convertido a CSV correctamente
